mlpi26_Lect02_lab.ipynb 

# Lecture 2 Lab — A Tour of Classifiers & Data Preprocessing
### 01211373 Machine Learning and Programming for Industry

| | |
|---|---|
| **Estimated time** | 2.5–3.5 hours |
| **Tools** | Python, NumPy, pandas, matplotlib, scikit-learn |
| **Submit** | This completed notebook (see §9) |

Type your name - surname and student ID in the cell below. Failure to do so resuls in -1 penalty for this lab.

In [ ]:
# your name - surname, student ID

## 1. Learning Objectives

By the end of this lab, you should be able to:

- Diagnose and handle missing values and categorical fields in a real-shaped sensor dataset.
- Build a scikit-learn `ColumnTransformer` + `Pipeline` that preprocesses numeric and categorical features together.
- Train and fairly compare four classifiers — logistic regression, k-NN, SVM, and a decision tree — on the same multiclass fault-detection task.
- Explain, in terms of how each algorithm works, why one classifier outperforms another on a given dataset.

## 2. Setup

Same environment as Week 1. Quick check that everything still imports:

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

print("Environment OK")

## 3. Part A — A Messier, Multiclass Dataset

This week's dataset has four classes instead of two — `Normal`, `Bearing Fault`, `Misalignment`, `Imbalance` — the same four summary features from Week 1 (`mean`, `std`, `rms`, `ptp`), plus two realistic complications:

- A **categorical** column, `machine_type` (`CNC` / `Press` / `Lathe`), which isn't a number and needs encoding.
- **Missing values**, injected into `std` and `rms`, simulating dropped sensor readings.

In [ ]:
def generate_fault_dataset(n_per_class=150, seed=0):
    rng = np.random.default_rng(seed)
    class_names = ["Normal", "Bearing Fault", "Misalignment", "Imbalance"] # 4 label types
    # rough per-class centers for (mean, std, rms, ptp)
    centers = {
        "Normal":        dict(mean=0.02, std=0.15, rms=0.20, ptp=0.60), # smooth periodic vibration
        "Bearing Fault": dict(mean=0.03, std=0.35, rms=0.45, ptp=1.80), # sharp periodic impulses from impact events
        "Misalignment":  dict(mean=0.25, std=0.20, rms=0.55, ptp=0.90), # elevated 2x running-speed harmonic
        "Imbalance":     dict(mean=0.05, std=0.18, rms=0.70, ptp=0.85), # elevated 1x running-speed amplitude
    }
    rows = []
    for label_idx, cname in enumerate(class_names): # generate data for each label, n_per_class each
        c = centers[cname]
        for _ in range(n_per_class):
            rows.append({
                "mean": rng.normal(c["mean"], 0.1), # original covariance 0.05
                "std": rng.normal(c["std"], 0.08), # original 0.04
                "rms": rng.normal(c["rms"], 0.12), # original 0.06
                "ptp": rng.normal(c["ptp"], 0.2), # original 0.15
                "machine_type": rng.choice(["CNC", "Press", "Lathe"]),  # randomly chosen
                "label": label_idx,
                "label_name": cname,
            })
    df = pd.DataFrame(rows)
    # inject ~5% missing values into two numeric columns "std","rms"
    for col in ["std", "rms"]:
        missing_idx = rng.choice(df.index, size=int(0.05 * len(df)), replace=False)
        df.loc[missing_idx, col] = np.nan
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)


df = generate_fault_dataset()
df.head()

## 4. Part B — Diagnose & Handle Missing Data

1. Check how many missing values each column has.
2. Decide: drop, impute, or flag-and-fill (see Week 1 slides §10 for the trade-offs)? For this lab, use **median imputation** on the numeric columns — sensor features like these are often skewed by occasional spikes, and the median is more robust to that than the mean.

In [ ]:
print(df.isna().sum())
print()
print(f"Total rows: {len(df)}")

> You don't need to hand-write the imputation here — it will happen inside the `Pipeline` in §6, so that it's correctly fit on the training set only. This step is just about *seeing* the problem before you solve it.

## 5. Part C — A First Look at Categorical Encoding

Before building the full pipeline, take a quick look at what one-hot encoding does to `machine_type`, using `pd.get_dummies` as a scratchpad (not what you'll use in the final pipeline — scikit-learn's `OneHotEncoder` handles this more robustly, see §6).

In [ ]:
pd.get_dummies(df[["machine_type"]]).head()

## 6. Part D — Build a Preprocessing Pipeline

Fill in the `TODO` below. You need a `ColumnTransformer` that:
- Applies **median imputation + standard scaling** to the numeric columns (`mean`, `std`, `rms`, `ptp`).
- Applies **one-hot encoding** to the categorical column (`machine_type`).

In [ ]:
numeric_features = ["mean", "std", "rms", "ptp"]
categorical_features = ["machine_type"]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = OneHotEncoder(handle_unknown="ignore")

# TODO: combine numeric_transformer and categorical_transformer into a single
# ColumnTransformer named `preprocessor`, applying numeric_transformer to
# numeric_features and categorical_transformer to categorical_features.
# Hint: ColumnTransformer(transformers=[(name, transformer, columns), ...])
preprocessor = ?

> **Why a `ColumnTransformer`?** Numeric and categorical columns need *different* preprocessing, but you still want everything to happen in one `.fit()` / `.transform()` call — on the training data only — so nothing leaks from the test set and nothing is easy to apply inconsistently. This is the Week 1 "fit scaler on train only" rule, generalized to a mix of column types.

## 7. Part E — Train & Compare Four Classifiers

Using the same `preprocessor`, fit all four classifiers from today's lecture on the same train/test split and compare their test accuracy.

In [ ]:
X = df[numeric_features + categorical_features]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y
)

# TODO : fill in the correct classfiers
classifiers = {
    "Logistic Regression": ?,
    "k-NN": ?,
    "SVM": ?,
    "Decision Tree": ?,
}

results = {}
fitted_pipelines = {}
for name, clf in classifiers.items():
    pipe = Pipeline(steps=[("preprocess", preprocessor), ("clf", clf)])
    pipe.fit(X_train, y_train)
    acc = accuracy_score(y_test, pipe.predict(X_test))
    results[name] = acc
    fitted_pipelines[name] = pipe
    print(f"{name:20s} accuracy: {acc:.3f}")

## 8. Part F — Compare & Interpret

1. Plot the four accuracies as a bar chart.
2. Pick the best-performing model and plot its confusion matrix.

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(results.keys(), results.values(), color=["#3E5C76", "#FF6A39", "#1B2A41", "#6B4F9C"])
plt.ylabel("Test accuracy")
plt.ylim(0, 1)
plt.title("Classifier comparison — multiclass fault detection")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
best_name = max(results, key=results.get)
best_pipe = fitted_pipelines[best_name]

ConfusionMatrixDisplay.from_estimator(
    best_pipe, X_test, y_test,
    display_labels=["Normal", "Bearing Fault", "Misalignment", "Imbalance"],
    cmap="Blues", xticks_rotation=20,
)
plt.title(f"Confusion matrix — {best_name}")
plt.tight_layout()
plt.show()

## 9. Reflection Questions

Answer briefly (2–3 sentences each) by editing the markdown cells below.

**1. Which classifier performed best, and does that match what you'd predict from how the algorithm works (see the comparison table in the Week 2 slides)?**

*Your answer:* 


**2. Why did we fit the imputer, scaler, and encoder only on the training set, inside the pipeline, rather than on the whole dataset upfront?**

*Your answer:*

**3. `machine_type` doesn't actually depend on which fault occurred in this synthetic dataset — it's assigned randomly. What would you expect to happen to accuracy if you dropped that column entirely? (Try it, if you're not sure.)**

*Your answer:*

**4. Suppose a fifth machine type appeared at test time that the encoder never saw during training. What does `handle_unknown="ignore"` in `OneHotEncoder` do in that case, and why is that safer than letting it raise an error?**

*Your answer:*

## 10. Deliverables & Submission

- This notebook, completed and able to run top-to-bottom without errors (`Kernel → Restart & Run All`).
- The bar chart and confusion matrix from §8.
- Your answers to the four reflection questions.

Submit this `.ipynb` file in google classroom before the due date. Late penalty is -1 per day.

## 11. Grading Rubric Guide

| Component | Weight |
|---|---|
| Missing data diagnosed and correctly handled | 15% |
| `ColumnTransformer` / `Pipeline` correctly built | 25% |
| All four classifiers trained & evaluated correctly | 25% |
| Interpretation of results (§8) and reflection questions | 25% |
| Notebook quality (runs cleanly top-to-bottom, reasonably organized) | 10% |

---
**Next week:** *Regression Analysis* — predicting continuous quantities, with an application to sensor calibration.

Generated by Claude and modified by dewdotninja

August 2026